# 02 — GPT-4o Fine-tuning for Skin Analysis

Fine-tune GPT-4o using OpenAI's vision fine-tuning API on our labeled skin dataset.

**Input:** Labeled images from `01_data_preparation.ipynb`
**Output:** Fine-tuned GPT-4o model that returns structured skin scores.

In [ ]:
!pip install -q openai pillow tqdm

In [ ]:
from google.colab import drive, userdata
drive.mount('/content/drive')

import os, json, base64, time
from tqdm import tqdm
from openai import OpenAI

OPENAI_API_KEY = userdata.get('OPENAI_API_KEY')
client = OpenAI(api_key=OPENAI_API_KEY)

DATA_DIR = '/content/drive/MyDrive/RadianceIQ/datasets/processed'
OUTPUT_DIR = '/content/drive/MyDrive/RadianceIQ/models/gpt4o'
os.makedirs(OUTPUT_DIR, exist_ok=True)

## 1. Prepare JSONL Training Data

In [ ]:
SYSTEM_PROMPT = """You are a dermatology analysis assistant. Given a photo of skin, analyze it and return structured scores.

Score each dimension 0-100 where 0 = no concern, 100 = severe:
- acne_score: inflammation, breakouts, comedones, papules, pustules
- sun_damage_score: hyperpigmentation, sunspots, melasma, UV damage
- skin_age_score: fine lines, wrinkles, texture roughness, elasticity loss

Also provide:
- confidence: "low", "med", or "high"
- primary_driver: main factor (e.g., "acne", "sun_damage", "texture")
- recommended_action: one actionable sentence

Return ONLY valid JSON."""

def image_to_base64(path):
    with open(path, 'rb') as f:
        return base64.b64encode(f.read()).decode('utf-8')

def create_training_example(item):
    img_b64 = image_to_base64(item['image_path'])
    scores = {'acne': item['acne_score'], 'sun_damage': item['sun_damage_score'], 'skin_age': item['skin_age_score']}
    primary = max(scores, key=scores.get)
    target = {
        'acne_score': item['acne_score'],
        'sun_damage_score': item['sun_damage_score'],
        'skin_age_score': item['skin_age_score'],
        'confidence': 'high',
        'primary_driver': primary,
        'recommended_action': f'Focus on {primary.replace("_", " ")} management for best results.',
    }
    return {
        'messages': [
            {'role': 'system', 'content': SYSTEM_PROMPT},
            {'role': 'user', 'content': [
                {'type': 'text', 'text': 'Analyze this skin photo and return structured scores.'},
                {'type': 'image_url', 'image_url': {'url': f'data:image/jpeg;base64,{img_b64}'}},
            ]},
            {'role': 'assistant', 'content': json.dumps(target)},
        ]
    }

In [ ]:
with open(f'{DATA_DIR}/train_augmented.json') as f:
    train_data = json.load(f)
with open(f'{DATA_DIR}/val.json') as f:
    val_data = json.load(f)

MAX_TRAIN, MAX_VAL = 500, 50
train_jsonl = f'{OUTPUT_DIR}/train.jsonl'
val_jsonl = f'{OUTPUT_DIR}/val.jsonl'

for path, data, limit in [(train_jsonl, train_data, MAX_TRAIN), (val_jsonl, val_data, MAX_VAL)]:
    with open(path, 'w') as f:
        for item in tqdm(data[:limit], desc=f'Creating {os.path.basename(path)}'):
            try:
                f.write(json.dumps(create_training_example(item)) + '\n')
            except Exception as e:
                print(f'Skipped: {e}')

print(f'Training JSONL: {train_jsonl}')
print(f'Validation JSONL: {val_jsonl}')

## 2. Upload & Launch Fine-tuning

In [ ]:
train_file = client.files.create(file=open(train_jsonl, 'rb'), purpose='fine-tune')
val_file = client.files.create(file=open(val_jsonl, 'rb'), purpose='fine-tune')
print(f'Training file: {train_file.id}, Validation file: {val_file.id}')

ft_job = client.fine_tuning.jobs.create(
    training_file=train_file.id,
    validation_file=val_file.id,
    model='gpt-4o-2024-08-06',
    hyperparameters={'n_epochs': 3, 'batch_size': 4, 'learning_rate_multiplier': 1.0},
    suffix='radianceiq-skin'
)
print(f'Job: {ft_job.id}, Status: {ft_job.status}')

## 3. Monitor Training

In [ ]:
start_time = time.time()
while True:
    job = client.fine_tuning.jobs.retrieve(ft_job.id)
    elapsed = (time.time() - start_time) / 60
    print(f'[{elapsed:.1f}m] Status: {job.status}')
    if job.status in ('succeeded', 'failed', 'cancelled'):
        break
    time.sleep(60)

if job.status == 'succeeded':
    print(f'Fine-tuned model: {job.fine_tuned_model}')
    with open(f'{OUTPUT_DIR}/model_info.json', 'w') as f:
        json.dump({'model_id': job.fine_tuned_model, 'job_id': ft_job.id, 'training_time_minutes': elapsed}, f, indent=2)
else:
    print(f'Job failed: {job.error}')

## 4. Test Fine-tuned Model

In [ ]:
with open(f'{DATA_DIR}/test.json') as f:
    test_data = json.load(f)

test_item = test_data[0]
img_b64 = image_to_base64(test_item['image_path'])

response = client.chat.completions.create(
    model=job.fine_tuned_model,
    messages=[
        {'role': 'system', 'content': SYSTEM_PROMPT},
        {'role': 'user', 'content': [
            {'type': 'text', 'text': 'Analyze this skin photo and return structured scores.'},
            {'type': 'image_url', 'image_url': {'url': f'data:image/jpeg;base64,{img_b64}'}},
        ]},
    ],
    max_tokens=300, temperature=0.2,
)

result = json.loads(response.choices[0].message.content)
print('Model output:', json.dumps(result, indent=2))
print(f'Ground truth: acne={test_item["acne_score"]}, sun={test_item["sun_damage_score"]}, age={test_item["skin_age_score"]}')